In [1]:
import pandas as pd

import pyecharts
from pyecharts import options as opts
from pyecharts.charts import Line

In [2]:
#%%  Reading CSV file into a DataFrame

avg_df = pd.read_csv('files\\avg_df.csv')

# Convert 'timestamp' to datetime format
avg_df['timestamp'] = pd.to_datetime(avg_df['timestamp'].str.split('/').str[0])

# Sort first by symbols (to group all records for each symbol together)
# Then by timestamp (descending) within each symbol group
avg_df = avg_df.sort_values(
    ['symbols', 'timestamp'], 
    ascending=[True, False]
).reset_index(drop=True)


In [8]:
# Create a line chart
line = Line(init_opts=opts.InitOpts(
    width="80%", 
    height="500px",
    theme="dark"  # Apply dark theme
))

# Get unique timestamps and format them for x-axis
timestamps = sorted(avg_df['timestamp'].unique())
x_data = [ts.strftime('%Y-%m-%d') for ts in timestamps]
line.add_xaxis(x_data)

# Add each symbol as a line
for symbol in avg_df['symbols'].unique():
    symbol_df = avg_df[avg_df['symbols'] == symbol].sort_values('timestamp')
    # Get price data aligned with the x-axis timestamps
    # This uses pandas' reindex to handle missing dates
    prices = symbol_df.set_index('timestamp').reindex(timestamps)['average_price'].tolist()
    line.add_yaxis(symbol, prices, is_symbol_show=False)

# Set global options
# Set global options
line.set_global_opts(
    title_opts=opts.TitleOpts(title="Cryptocurrency Price Trends"),
    tooltip_opts=opts.TooltipOpts(trigger="axis"),
    legend_opts=opts.LegendOpts(type_="scroll"),
    datazoom_opts=opts.DataZoomOpts(),
    xaxis_opts=opts.AxisOpts(
        type_="category", 
        axislabel_opts=opts.LabelOpts(rotate=30)
    ),
    yaxis_opts=opts.AxisOpts(
        type_="log",  # Set to log scale
        name="Price",
        name_location="middle",
        name_gap=35,
        splitline_opts=opts.SplitLineOpts(is_show=True),
        axisline_opts=opts.AxisLineOpts(is_show=True),
    ),
)

# Render the chart
line.render_notebook()